# Breast Cancer Detection — Notebook 5 (fast version)
## Swin Transformer on CBIS-DDSM Mammography

Fine-tunes **Swin-Tiny** (via `timm`) on full mammograms from **CBIS-DDSM** (mass + calcification cases). An image is labelled **malignant = 1** if any of its annotated abnormalities is MALIGNANT, otherwise 0. The official train/test split is respected.

**Fast settings (fits comfortably in one free-Colab session, ~15–20 min):**
- only the full-mammogram JPEGs are read from the zip (no full unzip)
- each mammogram is downscaled once to 224×224 and cached, so training never decodes the multi-megapixel originals again
- 6 epochs, mixed precision (AMP)

### 1. Setup

In [ ]:
!pip install -q timm

In [ ]:
import time
t_start = time.time()

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

base_path = "/content/drive/MyDrive/breast_cancer_project"

for folder in ["data", "models", "results", "plots"]:
    os.makedirs(os.path.join(base_path, folder), exist_ok=True)

print("Project folders ready")

In [ ]:
import os, shutil

# If kaggle.json is in the root of your Google Drive it is used automatically,
# so the notebook needs no further interaction after the Drive mount.
drive_token = "/content/drive/MyDrive/kaggle.json"
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)

if os.path.exists(drive_token):
    shutil.copy(drive_token, os.path.expanduser("~/.kaggle/kaggle.json"))
    print("kaggle.json copied from Drive")
else:
    from google.colab import files
    files.upload()   # upload kaggle.json
    shutil.copy("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))

os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

### 2. Dataset — CBIS-DDSM (JPEG release, ~5 GB download, no full unzip)

In [ ]:
!kaggle datasets download -d awsaf49/cbis-ddsm-breast-cancer-image-dataset

In [ ]:
import zipfile
import pandas as pd

zf = zipfile.ZipFile("cbis-ddsm-breast-cancer-image-dataset.zip")
names = zf.namelist()

def read_csv_from_zip(fname):
    match = [n for n in names if n.endswith("csv/" + fname)]
    return pd.read_csv(zf.open(match[0]))

dicom = read_csv_from_zip("dicom_info.csv")
mass_train = read_csv_from_zip("mass_case_description_train_set.csv")
mass_test = read_csv_from_zip("mass_case_description_test_set.csv")
calc_train = read_csv_from_zip("calc_case_description_train_set.csv")
calc_test = read_csv_from_zip("calc_case_description_test_set.csv")

print(dicom["SeriesDescription"].value_counts())
print("mass train/test:", len(mass_train), len(mass_test))
print("calc train/test:", len(calc_train), len(calc_test))

In [ ]:
# series UID -> zip member of its full-mammogram JPEG
jpeg_members = {n.split("/")[-2] + "/" + n.split("/")[-1]: n
                for n in names if n.lower().endswith(".jpg")}

full_mammo = dicom[dicom["SeriesDescription"] == "full mammogram images"]

uid_to_member = {}
for p in full_mammo["image_path"]:
    key = p.split("/")[-2] + "/" + p.split("/")[-1]
    if key in jpeg_members:
        uid_to_member[p.split("/")[-2]] = jpeg_members[key]

print("Full mammograms located in zip:", len(uid_to_member))
assert len(uid_to_member) > 0, "No full mammograms matched - check the zip layout above"

In [ ]:
def build_df(case_df):
    rows = []
    for _, r in case_df.iterrows():
        uid = r["image file path"].split("/")[-2]
        member = uid_to_member.get(uid)
        if member is not None:
            rows.append({
                "zip_name": member,
                "label": 1 if r["pathology"] == "MALIGNANT" else 0,
                "abnormality": r["abnormality type"],
            })
    frame = pd.DataFrame(rows)
    # one row per image: malignant if any abnormality on it is malignant
    return (frame.groupby("zip_name")
                 .agg(label=("label", "max"), abnormality=("abnormality", "first"))
                 .reset_index())

train_full_df = build_df(pd.concat([mass_train, calc_train], ignore_index=True))
test_df = build_df(pd.concat([mass_test, calc_test], ignore_index=True))

print("Train (official):", len(train_full_df), "| Test (official):", len(test_df))
print(train_full_df["label"].value_counts())

In [ ]:
import os
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

CACHE = "data/cbis_224"
os.makedirs(CACHE, exist_ok=True)
IMG_SIZE = 224

def cache_image(member):
    out = os.path.join(CACHE, member.replace("/", "_"))
    if not os.path.exists(out):
        with zipfile.ZipFile("cbis-ddsm-breast-cancer-image-dataset.zip") as z:
            with z.open(member) as fh:
                img = Image.open(fh)
                img.draft("L", (IMG_SIZE * 2, IMG_SIZE * 2))   # fast reduced-size JPEG decode
                img.convert("L").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR).save(out)
    return out

members = pd.concat([train_full_df["zip_name"], test_df["zip_name"]]).unique()
with ThreadPoolExecutor(max_workers=4) as pool:
    cached = dict(zip(members, pool.map(cache_image, members)))

train_full_df["path"] = train_full_df["zip_name"].map(cached)
test_df["path"] = test_df["zip_name"].map(cached)
zf.close()

print("Cached", len(cached), "mammograms at", IMG_SIZE, "x", IMG_SIZE)

In [ ]:
import random
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
from sklearn.model_selection import train_test_split

# carve a validation split out of the official training set
train_df, val_df = train_test_split(train_full_df, test_size=0.15,
                                    stratify=train_full_df["label"],
                                    random_state=42)

print("Train:", len(train_df), "| Val:", len(val_df), "| Test:", len(test_df))

In [ ]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(7),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
from torch.utils.data import Dataset, DataLoader

class MammogramDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")   # grayscale -> 3 channels
        label = row["label"]

        if self.transform:
            image = self.transform(image)

        return image, label

train_loader = DataLoader(MammogramDataset(train_df, train_transform),
                          batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(MammogramDataset(val_df, val_transform),
                        batch_size=64, num_workers=2, pin_memory=True)
test_loader = DataLoader(MammogramDataset(test_df, val_transform),
                         batch_size=64, num_workers=2, pin_memory=True)

img, label = MammogramDataset(train_df, train_transform)[0]
print(img.shape, label)

In [ ]:
import matplotlib.pyplot as plt

sample = pd.concat([train_df[train_df.label == 0].head(4),
                    train_df[train_df.label == 1].head(4)]).reset_index(drop=True)

fig, axes = plt.subplots(2, 4, figsize=(12, 7))
for ax, (_, row) in zip(axes.flat, sample.iterrows()):
    ax.imshow(Image.open(row["path"]), cmap="gray")
    ax.set_title(("MALIGNANT" if row["label"] == 1 else "BENIGN") +
                 f'\n({row["abnormality"]})', fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.savefig(base_path + "/plots/cbis_ddsm_samples.png", dpi=150, bbox_inches="tight")
plt.show()

### 3. Model — Swin-Tiny (`timm`) + AMP

In [ ]:
import timm
import torch.nn as nn

model = timm.create_model("swin_tiny_patch4_window7_224",
                          pretrained=True, num_classes=1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

n_pos = int(train_df["label"].sum())
n_neg = len(train_df) - n_pos
pos_weight = torch.tensor([n_neg / n_pos], device=device)
print("pos_weight:", round(pos_weight.item(), 3))

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=6)
scaler = torch.cuda.amp.GradScaler()   # mixed precision

### 4. Training (6 epochs, mixed precision)

In [ ]:
EPOCHS = 6
history = {"train_loss": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.float().to(device)

        with torch.cuda.amp.autocast():
            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        if i % 50 == 0:
            print(f"Epoch {epoch+1}, Step {i}, Loss: {loss.item():.4f}")
    scheduler.step()

    # ---------------- validation ----------------
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.float().to(device)

            outputs = model(images).squeeze(1)
            val_loss += criterion(outputs, labels).item()

            preds = (torch.sigmoid(outputs) > 0.5).int()
            correct += (preds == labels.int()).sum().item()
            total += labels.size(0)

    train_loss = total_loss / len(train_loader)
    val_loss = val_loss / len(val_loader)
    val_acc = correct / total

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), base_path + "/models/swin_cbis_ddsm_best.pth")
        print("  -> new best model saved")

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range, history["train_loss"], color="#0072B2", lw=2, label="Train loss")
axes[0].plot(epochs_range, history["val_loss"], color="#D55E00", lw=2, label="Val loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss curves")
axes[0].legend()

axes[1].plot(epochs_range, history["val_acc"], color="#009E73", lw=2, label="Val accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Validation accuracy")
axes[1].legend()

plt.tight_layout()
plt.savefig(base_path + "/plots/swin_cbis_ddsm_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### 5. Evaluation on the official CBIS-DDSM test split

In [ ]:
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, accuracy_score)

model.load_state_dict(torch.load(base_path + "/models/swin_cbis_ddsm_best.pth"))
model.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images).squeeze(1)
        probs = torch.sigmoid(outputs)

        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(np.array(labels))

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)
preds = (all_probs > 0.5).astype(int)

acc = accuracy_score(all_labels, preds)
auc = roc_auc_score(all_labels, all_probs)

print("Test Accuracy:", round(acc, 4))
print("ROC-AUC:", round(auc, 4))
print("\nConfusion Matrix:\n", confusion_matrix(all_labels, preds))
print("\nClassification Report:\n")
print(classification_report(all_labels, preds,
                            target_names=["benign", "malignant"]))

In [ ]:
import seaborn as sns
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Confusion matrix (single-hue sequential colormap)
cm = confusion_matrix(all_labels, preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["benign", "malignant"],
            yticklabels=["benign", "malignant"], ax=axes[0])
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].set_title("Confusion Matrix")

# ROC curve
fpr, tpr, _ = roc_curve(all_labels, all_probs)
axes[1].plot(fpr, tpr, color="#0072B2", lw=2, label=f"ROC (AUC = {auc:.3f})")
axes[1].plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--", label="Chance")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve")
axes[1].legend()

plt.tight_layout()
plt.savefig(base_path + "/plots/swin_cbis_ddsm_confusion_roc.png", dpi=150, bbox_inches="tight")
plt.show()

### 6. Save artifacts to Drive

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(all_labels, preds)
recall = recall_score(all_labels, preds)
f1 = f1_score(all_labels, preds)

results_text = f"""MODEL: Swin Transformer (Swin-Tiny, fast setting)
DATASET: CBIS-DDSM full mammograms (official test split)
LOSS: BCEWithLogitsLoss (pos_weight from data)

Test Accuracy: {acc:.4f}
ROC-AUC: {auc:.4f}

Malignant class:
Precision: {precision:.4f}
Recall: {recall:.4f}
F1-score: {f1:.4f}

Confusion Matrix:
{confusion_matrix(all_labels, preds)}

Classification Report:
{classification_report(all_labels, preds)}
"""

with open(base_path + "/results/swin_cbis_ddsm_results.txt", "w") as f:
    f.write(results_text)

print(results_text)
print("Results saved")

In [ ]:
results_df = pd.DataFrame({
    "path": test_df["path"].values,
    "true_label": all_labels,
    "predicted_label": preds,
    "probability": all_probs
})

results_df.to_csv(base_path + "/results/swin_cbis_ddsm_predictions.csv", index=False)
print("Predictions saved")

In [ ]:
import json

config = {
    "model": "swin_tiny_patch4_window7_224 (timm)",
    "pretrained": "ImageNet",
    "dataset": "CBIS-DDSM (awsaf49/cbis-ddsm-breast-cancer-image-dataset)",
    "images": "full mammograms (mass + calcification cases), cached at 224x224",
    "label": "1 if any abnormality on the image is MALIGNANT",
    "split": "official train/test, 15% of train held out for validation",
    "loss": "BCEWithLogitsLoss + pos_weight",
    "optimizer": "AdamW",
    "lr": 3e-5,
    "weight_decay": 0.05,
    "scheduler": "CosineAnnealingLR",
    "mixed_precision": True,
    "epochs": 6,
    "batch_size": 32,
    "image_size": 224,
    "threshold": 0.5,
    "seed": 42
}

with open(base_path + "/results/swin_cbis_ddsm_config.json", "w") as f:
    json.dump(config, f, indent=4)

print("Config saved")

In [ ]:
print(f"Total notebook time: {(time.time() - t_start) / 60:.1f} min")